In [13]:
import json
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
df = pd.read_csv(RAW / "Global_Landslide_Catalog_Export_rows.csv", dtype=str, keep_default_na=False)
print(len(df), "rows,", len(df.columns), "columns")
print(list(df.columns))

11033 rows, 31 columns
['source_name', 'source_link', 'event_id', 'event_date', 'event_time', 'event_title', 'event_description', 'location_description', 'location_accuracy', 'landslide_category', 'landslide_trigger', 'landslide_size', 'landslide_setting', 'fatality_count', 'injury_count', 'storm_name', 'photo_link', 'notes', 'event_import_source', 'event_import_id', 'country_name', 'country_code', 'admin_division_name', 'admin_division_population', 'gazeteer_closest_point', 'gazeteer_distance', 'submitted_date', 'created_date', 'last_edited_date', 'longitude', 'latitude']


In [14]:
blank = df.apply(lambda s: s.str.strip().eq(""))
prof = pd.DataFrame({"blank_pct": blank.mean() * 100, "distinct": df.nunique()})
prof.round(2)

,blank_pct,distinct
source_name,0.00,3918
source_link,7.67,8295
event_id,0.00,11033
event_date,0.00,6550
event_time,100.00,1
event_title,0.00,10549
event_description,7.81,9410
location_description,0.92,10436
location_accuracy,0.02,10
landslide_category,0.01,15


In [15]:
def shapes(s, n=10):
    return (s.str.replace(r"\d", "9", regex=True)
             .str.replace(r"[A-Za-z]+", "a", regex=True).value_counts().head(n))

for c in ["event_date", "event_time", "last_edited_date", "location_accuracy",
          "fatality_count", "injury_count"]:
    if c in df:
        print(c, "\n", shapes(df[c]), "\n")

event_date 
 event_date
99/99/9999 99:99:99 a    11033
Name: count, dtype: int64 

event_time 
 event_time
    11033
Name: count, dtype: int64 

last_edited_date 
 last_edited_date
99/99/9999 99:99:99 a    11033
Name: count, dtype: int64 

location_accuracy 
 location_accuracy
9a      5363
99a     3699
a       1928
999a      41
           2
Name: count, dtype: int64 

fatality_count 
 fatality_count
9       9197
        1385
99       418
999       30
9999       3
Name: count, dtype: int64 

injury_count 
 injury_count
       5674
9      5281
99       72
999       6
Name: count, dtype: int64 



In [16]:
for c in ["landslide_category", "landslide_trigger", "landslide_size",
          "landslide_setting", "location_accuracy"]:
    print(df[c].value_counts(dropna=False).to_string(), "\n")

landslide_category
landslide              7648
mudslide               2100
rock_fall               671
complex                 232
debris_flow             194
other                    68
unknown                  38
riverbank_collapse       37
snow_avalanche           15
translational_slide       9
lahar                     7
earth_flow                7
creep                     5
                          1
topple                    1 

landslide_trigger
downpour                   4680
rain                       2592
unknown                    1691
continuous_rain             748
tropical_cyclone            561
snowfall_snowmelt           135
monsoon                     129
mining                       93
earthquake                   89
construction                 82
flooding                     75
no_apparent_trigger          44
freeze_thaw                  41
other                        26
                             23
dam_embankment_collapse      12
leaking_pipe                 

In [17]:
lat = pd.to_numeric(df.latitude, errors="coerce")
lng = pd.to_numeric(df.longitude, errors="coerce")
print("non-numeric lat/lng:", lat.isna().sum(), lng.isna().sum())
print(lat.describe(), lng.describe(), sep="\n")
print("(0,0):", ((lat == 0) & (lng == 0)).sum(),
      "| out of range:", (~lat.between(-90, 90) | ~lng.between(-180, 180)).sum())

non-numeric lat/lng: 0 0
count    11033.000000
mean        25.881887
std         20.415054
min        -46.774800
25%         13.917600
50%         30.534500
75%         40.866259
max         72.627500
Name: latitude, dtype: float64
count    11033.000000
mean         2.520441
std        100.908393
min       -179.980766
25%       -107.871700
50%         19.694600
75%         93.948000
max        179.991364
Name: longitude, dtype: float64
(0,0): 0 | out of range: 0


In [18]:
ids = df.event_id.str.strip()
dups = df[ids.duplicated(keep=False) & ids.ne("")]
print("blank ids:", ids.eq("").sum(), "| rows in duplicate groups:", len(dups))
dups.sort_values("event_id").head(10)

blank ids: 0 | rows in duplicate groups: 0


,source_name,source_link,event_id,event_date,event_time,event_title,event_description,location_description,location_accuracy,landslide_category,...,country_code,admin_division_name,admin_division_population,gazeteer_closest_point,gazeteer_distance,submitted_date,created_date,last_edited_date,longitude,latitude


In [19]:
print({c: int(df[c].str.contains(r"[\r\n]").sum())
       for c in ["event_title", "event_description", "location_description"]})

{'event_title': 0, 'event_description': 7, 'location_description': 0}


In [20]:
rows = ["| column | blank % | distinct | examples |", "|---|---|---|---|"]
for c in df.columns:
    ex = ", ".join(df[c][df[c].str.strip() != ""].drop_duplicates().head(3).str[:30])
    rows.append(f"| `{c}` | {prof.blank_pct[c]:.1f} | {prof.distinct[c]} "
                f"| {ex.replace('|', '/')} |")
Path("../docs").mkdir(exist_ok=True)
Path("../docs/data_dictionary.md").write_text("\n".join(rows) + "\n")

2448